# 00 データと基準 — 「観測できるもの」と「本当の好み」は、違う

> 🔒 すべて**合成データ**です。実在の個人の閲覧・購入履歴は使いません。

## ゴール
- 推薦のデータ（**誰が何を評価したか**）が、**偏っている**ことを、数字で確かめる。
- **ランダム／人気順／平均評価順**という、簡単な**基準**を作る。
- **観測データでの評価**と、**本当の好みでの評価**が、**食い違う**ことを体験する。

## 推薦とは
ユーザーに、**好みそうな商品**を薦めます。分類や予測と似ていますが、次の点が**違います**。

- **欲しい答えは「順位」**: 上位10件に、好きなものが入っているか。
- **見えているデータは偏っている**: 人は、**人気のもの**や**好きなもの**に触れやすい。評価されていない商品は、**嫌い**なのではなく、**知らない**だけかもしれない。
- だから、**観測データでの評価が高い＝本当に良い推薦**、とは限らない。

## 合成データなら、「本当の好み」が分かる
このプロジェクトでは、**本当の好み**（全ユーザー×全商品）を持つ世界を作り、そこから**偏った観測**（誰が何を評価したか）を作ります。
手法には**観測だけ**を渡し、評価では、**観測テスト**と**本当の好み**の両方で測ります。

## キーワード
- **評価（rating）**: ユーザーが商品につけた点（1〜5）。**関連あり（好き）**は、4以上。
- **precision@10**: 上位10件のうち、好きなものの割合。**NDCG@10**: 好きなものが上位にあるほど高い（0〜1）。
- **露出（exposure）の偏り**: 人気のものほど、目に触れ、評価されやすい。
- **カバレッジ**: 薦められた商品の種類の割合。**ジニ係数**: 薦める回数の偏り（0＝均等、1＝1つに集中）。
- **人気順**: 評価された回数が多い商品から薦める。全員に同じもの。

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from recsys_synth import make_dataset
from recsys_eval import (evaluate_observed, evaluate_true, exclusion_mask, topk_items, beyond_accuracy, gini,
                         random_scores, popularity_scores, item_mean_scores, oracle_scores)

plt.rcParams["font.family"] = ["Hiragino Sans", "AppleGothic", "DejaVu Sans"]

d = make_dataset(1_000_000)
w, df = d.world, d.interactions
U, I = w.n_users, w.n_items
print("観測データの先頭:")
display(df.head(5))
print("商品の特徴:")
display(d.items.head(3))
summary = {
    "ユーザー数": U, "商品数": I, "評価の数": len(df), "密度（評価された組の割合）": round(len(df) / (U * I), 3),
    "train / val / test の件数": f"{(df.split == 'train').sum()} / {(df.split == 'val').sum()} / {(df.split == 'test').sum()}",
}
pd.Series(summary, name="値").to_frame()

**何をしているか**: 合成データを作ります。手法に渡してよいのは、**観測**（`user`・`item`・`rating`・`time`・`split`）と、商品の**ジャンル**だけです。本当の好みは、手元の `w`（世界）にありますが、**評価にだけ使います**。

**出力の見方**:
- 1000人のユーザー・500商品で、評価は約2.5万件。**密度は約5%**（組み合わせの95%は、評価されていない）。推薦の多くのデータは、**とても疎**です。
- データは**時間順**に、train（時刻0.7まで）・val（0.7〜0.85）・test（0.85以降）に分けます（未来を学習に使わない）。

In [ ]:
tr = d.train()
cnt = tr.groupby("item").size().reindex(range(I), fill_value=0).sort_values(ascending=False).to_numpy()
act = tr.groupby("user").size().reindex(range(U), fill_value=0).to_numpy()
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
bins = np.arange(0.5, 6.5, 1)
ax[0].hist(df["rating"], bins=bins, alpha=0.6, density=True, label="観測された評価")
ax[0].hist(np.clip(np.round(w.true_rating()).ravel(), 1, 5), bins=bins, alpha=0.6, density=True, label="本当の好み（全部の組）")
ax[0].set(title="評価の分布", xlabel="評価（1〜5）"); ax[0].legend(fontsize=8)
ax[1].plot(cnt); ax[1].set(title="商品の人気（評価された回数）", xlabel="人気の順位", ylabel="回数（train）")
ax[2].hist(act, bins=30); ax[2].set(title="ユーザーごとの評価の数", xlabel="件数（train）", ylabel="ユーザー数")
for a in ax: a.grid(alpha=0.3)
fig.tight_layout()
print(f"人気上位10%（50商品）が、学習データの {cnt[:50].sum() / cnt.sum():.1%} を占める。ジニ係数 {gini(cnt):.2f}")
print(f"評価が0回の商品: {(cnt == 0).sum()}個 ／ ユーザーあたりの評価数の中央値: {int(np.median(act))}件")
print(f"観測された評価の平均 {df['rating'].mean():.2f} ／ 全部の組の、本当の好みの平均 {w.true_rating().mean():.2f}")
print(f"4以上の割合: 観測 {(df['rating'] >= 4).mean():.1%} ／ 本当の好み {(w.true_rating() >= 4).mean():.1%}")

**何をしているか**: データの偏りを、3つの図で見ます。

**出力の見方**:
- **左（評価の分布）**: 観測された評価（青）は、**高い評価に偏っています**。観測の平均は3.6、4以上が55%。一方、**全部の組の本当の好み**（橙）は、平均3.3で、4以上は26%。**人は、好きなものを評価しやすい**ので、観測は**実際より好意的**に見えます。
- **中（人気）**: 少数の商品に、評価が**集中**します（**人気上位10%の商品が、43%**を占める。ジニ係数0.57）。大部分の商品は、ほとんど評価されません（**ロングテール**）。
- **右（ユーザー）**: 評価の数は、ユーザーによって大きく違います（中央値12件、最大154件）。少ししか評価していないユーザーが多く、**好みを推定しにくい**。

In [ ]:
true = w.true_rating()
observed = np.zeros((U, I), dtype=bool)
observed[df["user"].to_numpy(), df["item"].to_numpy()] = True        # 全期間で、評価された組
rows = []
for lo, hi in [(1, 2.5), (2.5, 3.5), (3.5, 4.5), (4.5, 5.01)]:
    m = (true >= lo) & (true < hi)
    rows.append({"本当の好み": f"{lo}〜{hi:.1f}", "組の数": int(m.sum()), "評価された割合": round(float(observed[m].mean()), 4)})
by_taste = pd.DataFrame(rows).set_index("本当の好み")
pop_rank = np.argsort(np.argsort(-w.item_pop))                         # 人気の順位（0が一番人気）
by_pop = pd.DataFrame([{"人気（上位から）": f"{k * 20}〜{(k + 1) * 20}%", "評価された割合": round(float(observed[:, (pop_rank >= k * I // 5) & (pop_rank < (k + 1) * I // 5)].mean()), 4)} for k in range(5)]).set_index("人気（上位から）")
display(by_taste); display(by_pop)

**何をしているか**: **どんな組が、評価されやすいか**を調べます。上の表は「本当の好みの高さ」ごと、下の表は「商品の人気の順位」ごとの、**評価された割合**です。

**出力の見方**:
- **好きなものほど、評価されやすい**: 本当の好みが4.5以上の組は、1〜2.5の組より、**評価される割合が約2.6倍**（7.6% 対 2.9%）。
- **人気の商品ほど、評価されやすい**: 人気上位20%の商品の、評価された割合（14.8%）は、下位20%（0.65%）の**約23倍**。
- つまり、**評価されていない＝嫌い、ではありません**。**人気がなくて、触れられなかった**だけかもしれません（**知らない**）。観測データだけを見ると、**人気のもの・好きなものの情報ばかり**が集まります。

In [ ]:
tr, val = d.split("train"), d.split("val")
excl = exclusion_mask(U, I, tr)                         # すでに評価した商品は、薦めない
pop_tr = tr.groupby("item").size().reindex(range(I), fill_value=0).to_numpy()
rows = []
for name, S in [("ランダム", random_scores(U, I)), ("人気順", popularity_scores(tr, U, I)),
                ("人気順（高評価の回数）", popularity_scores(tr, U, I, liked_only=True)), ("平均評価順", item_mean_scores(tr, U, I)),
                ("上限（本当の好みで薦める）", oracle_scores(w.true_rating()))]:
    tk = topk_items(S, excl, 10)
    ob, tt = evaluate_observed(S, tk, val, U, I), evaluate_true(tk, w.true_rating(), excl)
    ba = beyond_accuracy(tk, I, pop_tr, w.item_genre)
    rows.append({"手法": name, "観測テスト precision@10": round(ob["precision@10"], 4), "観測テスト NDCG@10": round(ob["ndcg@10"], 4),
                 "本当の好み precision@10": round(tt["precision@10"], 3), "カバレッジ": round(ba["カバレッジ"], 2),
                 "露出のジニ": round(ba["露出のジニ係数"], 2)})
base = pd.DataFrame(rows).set_index("手法")
print("評価したユーザー数（観測テスト）:", ob["評価したユーザー数"])
base

## 基準の比較 — 観測テストと、本当の好みで、順位が逆転する

**何をしているか**: 5つの簡単な方法で、各ユーザーに10件を薦めます。**観測テスト**（val期間に、実際に高く評価した商品に当たったか）と、**本当の好み**（本当に好きな商品に当たったか）の、**両方**で評価します。
「**上限**」は、**本当の好みで薦める**理想の推薦（現実には作れません）です。

**出力の見方**:
- **観測テストでは、人気順が最も良い**: precision@10 は **0.074**。**上限（0.023）より、ずっと高い**。ランダム（0.008）の約10倍。
  - 観測データは、**人気のものほど評価されやすい**ので、**人気のものを薦めれば、観測と一致しやすい**のです。本当に好きなもの（人気は低いかもしれない）を薦めても、観測テストには、あまり出てきません。
- **本当の好みで見ると、順位が逆転**: **上限0.97 ＞ 平均評価順0.65 ＞ 人気順（高評価）0.46 ＞ 人気順0.35 ＞ ランダム0.26**。**人気順は、本当の好みへの当たりは低い**（ランダムは、関連ありの割合26%が基準）。
- **人気順は、全員にほぼ同じ商品を薦めます**: カバレッジ0.06（500商品のうち、薦めたのは約6%）、露出のジニ0.97。
- 結論: **観測テストだけで評価すると、人気順を「最強」と誤解する**。**オフライン評価は、偏った観測に基づく**ことを、忘れない。

In [ ]:
rows = []
for pp, te in [(0.0, 0.5), (0.5, 0.5), (1.0, 0.5), (1.5, 0.5)]:
    dd = make_dataset(1_000_000, pop_power=pp, taste_effect=te)
    ww = dd.world
    t_, v_ = dd.split("train"), dd.split("val")
    ex = exclusion_mask(ww.n_users, ww.n_items, t_)
    r = {"人気の効き pop_power": pp}
    for name, S in [("人気順", popularity_scores(t_, ww.n_users, ww.n_items)), ("上限（本当の好み）", oracle_scores(ww.true_rating()))]:
        r[f"観測NDCG {name}"] = round(evaluate_observed(S, topk_items(S, ex, 10), v_, ww.n_users, ww.n_items)["ndcg@10"], 4)
    rows.append(r)
pd.DataFrame(rows).set_index("人気の効き pop_power")

**何をしているか**: データを作るときの**人気の効き**（`pop_power`。大きいほど、人気の商品ばかりが評価される）を変えて、**観測テストでの**人気順と上限の、NDCG@10を比べます。

**出力の見方**:
- **人気の効き＝0（偏りなし）**: **上限（0.057）が、人気順（0.025）を上回ります**。偏りがなければ、本当の好みで薦めるのが、観測でも当たる。
- **0.5**: 人気順（0.070）が、上限（0.051）を**逆転**。
- **1.0**（標準）: 人気順 **0.198** ＞ 上限 0.048。**1.5**: 人気順 **0.304**、上限 0.043。
- **偏りが強いほど、観測テストは「人気順が強い」と錯覚させます**。現実のデータの偏りの強さは分からないので、**観測テストの数字を、そのまま信じてはいけません**。次の Phase1 で、さらに調べます。

## 試してみよう（演習）
1. **人気の効きを変える**: 最後のセルの `(0.5, 0.5)` を `(0.25, 0.5)` や `(0.75, 0.5)` に変えて、人気順と上限の**逆転する境目**を探してください。分かるのは、偏りが少しでもあると、観測テストの順位が変わることです。
2. **好みの効きを変える**: `make_dataset(..., taste_effect=0.5)` の `0.5` を `0.0`（好みと関係なく選ぶ）と `1.0` にして、「観測の4以上の割合」（上の図のセル）がどう変わるか見てください。分かるのは、好きなものほど評価される度合いで、観測が好意的になる度合いが決まることです。
3. **k を変える**: `topk_items(S, excl, 10)` の `10` を `5` と `20` にして、基準の順位が変わるか見てください。
4. **別の世界で試す**: `make_dataset(1_000_000)` を `make_dataset(1_000_001)` に変えて、同じ結論か確かめてください。分かるのは、世界（シード）が変わっても、傾向が同じかどうかです。

## まとめ
- 推薦のデータは、**とても疎**（密度5%）で、**人気に偏る**（上位10%の商品が43%）。評価されていないのは、**嫌い**でなく**知らない**かもしれない。
- 観測された評価は、**好意的**に偏る（4以上が55%。本当は26%）。
- **観測テストでは、人気順が最強に見える**（上限より高い）。**本当の好みで見ると、順位が逆転**し、人気順は本当の好みへの当たりが低い。偏りが強いほど、この錯覚は強くなる。
- 人気順は、**全員にほぼ同じ商品**を薦める（カバレッジ6%）。精度だけで選ばない。
- 以降、**観測テストと、本当の好みの、両方**で評価する。

次の Phase1 では、評価の落とし穴（RMSEと順位の違い、評価の候補の選び方）を、さらに調べます。